[< Back to Main README](../README.md) | [Demo README](./README.md)

# RAG vs Graph-RAG: Reducing Agent Hallucinations

**Research Validated:**
- [Internal Representations as Indicators of Hallucinations](https://arxiv.org/pdf/2601.05214)
- [RAG-KG-IL: Multi-Agent Hybrid Framework](https://arxiv.org/pdf/2503.13514)
- [MetaRAG: Metamorphic Testing for Hallucination Detection](https://arxiv.org/pdf/2509.09360)

---

## What We're Testing

| Test | What It Measures | RAG Expected | Graph-RAG Expected |
|------|-----------------|--------------|--------------------|
| Aggregation | Can it compute averages? | ❌ Guesses | ✅ Native AVG() |
| Counting | Can it count across docs? | ❌ Can't | ✅ Native COUNT() |
| Multi-hop | Can it traverse relations? | ❌ Limited | ✅ Cypher traversal |
| Out-of-domain | Does it hallucinate? | ❌ Fabricates | ✅ Honest failure |

---

## Configure AWS Credentials

This demo uses Amazon Bedrock (default model provider for Strands Agents). Ensure your AWS credentials are configured.

To use a different provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).

In [ ]:
# Ensure AWS region is set (required for Bedrock in Workshop Studio)
import os
if not os.environ.get("AWS_DEFAULT_REGION") and not os.environ.get("AWS_REGION"):
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

import os
# Verify AWS credentials are available
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("\u2705 AWS credentials configured")

# To use OpenAI instead of Bedrock:
#   pip install "strands-agents[openai]"
#   os.environ["OPENAI_API_KEY"] = "your-key-here"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")


## Setup

In [ ]:
import os
os.environ['OTEL_SDK_DISABLED'] = 'true'
from dotenv import load_dotenv
load_dotenv()

from strands import Agent, tool
from neo4j import GraphDatabase
import faiss
import json
import boto3
import numpy as np

# Get Neo4j credentials from environment variables
# Workshop Studio: Pre-configured in /etc/environment by CloudFormation
# Self-paced: Set NEO4J_URI, NEO4J_USERNAME and NEO4J_PASSWORD before running
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

if not NEO4J_URI or not NEO4J_USERNAME or not NEO4J_PASSWORD:
    raise ValueError(
        'Neo4j credentials not found.\n'
        'Expected environment variables: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD\n'
        'Workshop Studio: Check that CloudFormation deployment completed successfully.\n'
        'Self-paced: Set these variables before running the notebook.'
    )

print(f"✅ Neo4j URI: {NEO4J_URI}")
print(f"✅ Neo4j password: {'*' * len(NEO4J_PASSWORD)} ({len(NEO4J_PASSWORD)} chars)")

# Amazon Bedrock Nova 2 for embeddings (no SentenceTransformer / OpenAI needed)
_bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

def _embed(text):
    """Embed text using Amazon Bedrock Nova 2 Multimodal Embeddings."""
    resp = _bedrock.invoke_model(
        modelId="amazon.nova-2-multimodal-embeddings-v1:0",
        body=json.dumps({
            "taskType": "SINGLE_EMBEDDING",
            "singleEmbeddingParams": {
                "embeddingPurpose": "GENERIC_INDEX",
                "embeddingDimension": 1024,
                "text": {"truncationMode": "END", "value": text[:8000]},
            },
        }),
        contentType="application/json",
        accept="application/json",
    )
    result = json.loads(resp["body"].read())
    return np.array([result["embeddings"][0]["embedding"]], dtype="float32")

# Build FAISS index if not already built
import subprocess, sys, os
from pathlib import Path

def build_faiss_if_needed():
    full_index = "faqs_vector.index"
    full_docs = "faqs_docs.json"
    
    if os.path.exists(full_index) and os.path.exists(full_docs):
        return full_index, full_docs
    
    # Extract hotel FAQ data if not already extracted
    data_dir = "data"
    if not os.path.exists(data_dir) or not list(Path(data_dir).glob("*.txt")):
        zip_file = "hotel-faqs.zip"
        if not os.path.exists(zip_file):
            raise FileNotFoundError(
                f"hotel-faqs.zip not found in {os.getcwd()}.\n"
                "At an AWS Event: the data is pre-loaded — check that you cd to the right folder.\n"
                "Self-paced: download hotel-faqs.zip from the workshop releases page."
            )
        print(f"Extracting {zip_file}...")
        import zipfile
        with zipfile.ZipFile(zip_file, "r") as zf:
            zf.extractall(data_dir)
        print(f"Extracted {len(list(Path(data_dir).glob('*.txt')))} hotel FAQ documents.")
    
    print("Building FAISS index from hotel FAQ data (full: 300 docs, ~10 min)...")
    result = subprocess.run([sys.executable, "load_vector_data.py"], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Failed to build FAISS index:\n{result.stderr}")
    print("FAISS index built.")
    return full_index, full_docs

index_file, docs_file = build_faiss_if_needed()

# Load FAISS
index = faiss.read_index(index_file)
with open(docs_file, "r", encoding="utf-8") as f:
    documents = json.load(f)
print(f"✅ FAISS: {len(documents)} documents")

# Check Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
with driver.session() as session:
    # Measured once here and reused by the test cells. No corpus size is
    # hardcoded anywhere in this notebook — a rebuilt graph changes the number
    # and every claim below follows it.
    HOTEL_COUNT = session.run('MATCH (h:Hotel) RETURN count(h) as c').single()['c']
    print(f"✅ Neo4j: {HOTEL_COUNT} hotels in knowledge graph")
driver.close()

## Define Tools & Agents

In [ ]:
# Tool observability hook - captures tool inputs and outputs
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import BeforeToolCallEvent, AfterToolCallEvent

class ToolObservabilityHook(HookProvider):
    """Display tool inputs and outputs for debugging and comparison.
    
    Uses Strands hook system to intercept tool execution:
    - BeforeToolCallEvent: captures tool name and input parameters
    - AfterToolCallEvent: captures tool result and status
    
    This approach keeps tools clean (no logging code inside them)
    and provides a reusable observability layer.
    """
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.on_before_tool)
        registry.add_callback(AfterToolCallEvent, self.on_after_tool)
    
    def on_before_tool(self, event: BeforeToolCallEvent) -> None:
        """Called before tool execution - shows input parameters."""
        tool_input = event.tool_use.get("input", {})
        
        print(f"   📥 Tool Input:")
        for key, value in tool_input.items():
            val_str = str(value)
            # Truncate long values (e.g., Cypher queries)
            if len(val_str) > 300:
                val_str = val_str[:300] + "..."
            print(f"      {key}: {val_str}")
    
    def on_after_tool(self, event: AfterToolCallEvent) -> None:
        """Called after tool execution - shows output/result."""
        status = event.result["status"]
        
        # Handle errors
        if status == "error" or event.exception:
            print(f"   ❌ Error: {event.exception}")
            return
        
        # Extract content from result
        if event.result["content"]:
            for content_item in event.result["content"]:
                # Most common: text output
                if "text" in content_item:
                    result_text = content_item["text"]
                    # Show first 800 chars of output
                    if len(result_text) > 800:
                        print(f"   📤 Tool Output (truncated):\n{result_text[:800]}...")
                    else:
                        print(f"   📤 Tool Output:\n{result_text}")
                # Handle JSON output
                elif "json" in content_item:
                    print(f"   📤 Tool Output (JSON):\n{content_item['json']}")

print("✅ Tool observability hook defined")

In [ ]:
@tool
def search_faqs(query: str) -> str:
    """Search hotel FAQs using vector similarity (Traditional RAG)."""
    query_embedding = _embed(query)
    distances, indices = index.search(query_embedding, 3)
    results = []
    for idx in indices[0]:
        doc = documents[idx]
        results.append(f"[{doc['filename']}]\n{doc['text'][:500]}...")
    return "\n\n".join(results)

@tool
def query_knowledge_graph(cypher_query: str) -> str:
    """Execute a Cypher query against the hotel knowledge graph.
    
    Node labels: Hotel, Room, Amenity, Policy, Service
    Hotel properties: name, address, guest_rating, total_rooms, email, phone
    Room properties: type, bed_configuration, max_occupancy, min_rate, max_rate
    Amenity properties: name, description, fee
    Policy properties: name, description
    Service properties: name, description, cost, hours, is_available, is_complimentary
    
    Relationships: (Hotel)-[:HAS_ROOM]->(Room), (Hotel)-[:OFFERS_AMENITY]->(Amenity),
                   (Hotel)-[:HAS_POLICY]->(Policy), (Hotel)-[:PROVIDES_SERVICE]->(Service)
    
    Location is in Hotel.address property. Use: WHERE h.address CONTAINS 'Cairo'
    IMPORTANT: All property names use snake_case (e.g., guest_rating NOT guestRating)
    """
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    with driver.session() as session:
        try:
            result = session.run(cypher_query)
            records = list(result)
            if not records:
                return "No results found."
            output = f"Found {len(records)} results:\n"
            for record in records[:15]:
                output += f"  {dict(record.items())}\n"
            return output
        except Exception as e:
            return f"Query error: {str(e)}"
        finally:
            driver.close()

# Model configuration — Amazon Bedrock (default, no extra import needed)
# Strands Agents uses Bedrock by default when no model is specified.
#
# To use OpenAI instead:
#   pip install "strands-agents[openai]"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")
#   rag_agent = Agent(..., model=MODEL)
#
# See all providers: https://strandsagents.com/docs/user-guide/concepts/model-providers/

# Create observability hook instance
observability_hook = ToolObservabilityHook()

rag_agent = Agent(
    name="RAG_Agent",
    system_prompt="You are a travel agent. Use vector search to find relevant FAQ information.",
    tools=[search_faqs],
    hooks=[observability_hook],
)

graph_agent = Agent(
    name="GraphRAG_Agent",
    system_prompt="You are a travel agent. Use the knowledge base to answer questions accurately. You can run multiple queries.",
    tools=[query_knowledge_graph],
    hooks=[observability_hook],
)

print("✅ Agents ready")

In [ ]:
# Per-query token accounting
#
# Both agents are created once and reused across every query below. Strands
# exposes token usage as agent.event_loop_metrics.accumulated_usage (the same
# figure surfaced on result.metrics.accumulated_usage), and that counter is a
# LIFETIME total that only ever grows. Reading it raw after each query makes
# later queries look far more expensive than they were and distorts the
# headline RAG vs Graph-RAG comparison.
#
# Fix: snapshot the counter before a query and subtract after, so each query
# reports the tokens used FOR THAT QUERY.
_USAGE_KEYS = ("inputTokens", "outputTokens", "totalTokens")

def usage_snapshot(agent) -> dict:
    """Read an agent's lifetime accumulated token usage."""
    metrics = getattr(agent, "event_loop_metrics", None)
    if metrics is None:
        return dict.fromkeys(_USAGE_KEYS, 0)
    usage = metrics.accumulated_usage
    return {k: usage.get(k, 0) for k in _USAGE_KEYS}

def usage_delta(agent, before: dict) -> dict:
    """Tokens used by the call that just ran, given a pre-call snapshot."""
    after = usage_snapshot(agent)
    return {k: after[k] - before.get(k, 0) for k in _USAGE_KEYS}

print("✅ Per-query token accounting ready")

---

## Test 1: Aggregation

**Paper:** "RAG cannot compute aggregations — LLM guesses from text chunks"

**Query:** What is the average guest rating across all hotels in Paris?

### Token Counting with Strands

Strands Agents provides **native token counting** through `agent.event_loop_metrics.accumulated_usage` (also surfaced on `AgentResult.metrics.accumulated_usage`). One catch: that counter is a **lifetime total** for the agent, so it keeps growing as the same agent answers query after query.

Since both agents are reused across every test below, reading the counter raw would make each later query look more expensive than it really was. Instead, snapshot before the query and subtract after to get the tokens used **for that query**:

```python
before = usage_snapshot(rag_agent)
result = rag_agent("Your query")
usage = usage_delta(rag_agent, before)
print(f"Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")
```

Available metrics:
- `inputTokens` — Tokens sent to the model
- `outputTokens` — Tokens generated by the model  
- `totalTokens` — Total (input + output)
- `cacheReadInputTokens` — Tokens read from cache (when using prompt caching)
- `cacheWriteInputTokens` — Tokens written to cache (when using prompt caching)

Token counts shown below each agent response reflect that single query and demonstrate cost differences between approaches.

In [ ]:
query = "What is the average guest rating of all hotels in Paris?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
before = usage_snapshot(rag_agent)
r = rag_agent(query)

if r.metrics:
    usage = usage_delta(rag_agent, before)
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

In [ ]:
query = "What is the average guest rating of all hotels in Paris?"

print("[GRAPH-RAG]")
before = usage_snapshot(graph_agent)
r = graph_agent(query)

if r.metrics:
    usage = usage_delta(graph_agent, before)
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 RAG can only average the hotels its top-3 retrieval surfaced | Graph-RAG runs AVG() over every Paris hotel")

In [ ]:
query = "How many hotels in the database have a swimming pool?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
before = usage_snapshot(rag_agent)
r = rag_agent(query)

if r.metrics:
    usage = usage_delta(rag_agent, before)
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

In [ ]:
query = "How many hotels in the database have a swimming pool?"

print("[GRAPH-RAG]")
before = usage_snapshot(graph_agent)
r = graph_agent(query)

if r.metrics:
    usage = usage_delta(graph_agent, before)
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

# HOTEL_COUNT is measured from the graph in the setup cell, never hardcoded.
# The claim worth making is "Graph-RAG counts the whole corpus", and that stays
# true for any corpus size — but only if the number is read back rather than
# asserted.
print(f"\n📊 RAG only sees top 3 docs, cannot count | Graph-RAG executes COUNT() across all {HOTEL_COUNT} hotels in the graph")

---

## Test 4: Out-of-Domain Detection

**Paper:** "RAG hallucinates when data doesn't exist — returns plausible but fabricated answers"

**Query:** Tell me about hotels in Antarctica

In [ ]:
query = "Tell me about hotels in Antarctica"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
before = usage_snapshot(rag_agent)
r_rag = rag_agent(query)

if r_rag.metrics:
    usage = usage_delta(rag_agent, before)
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

In [ ]:
query = "Which hotels in Cairo have both a spa and a swimming pool, and what are their guest ratings?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
before = usage_snapshot(rag_agent)
r = rag_agent(query)

if r.metrics:
    usage = usage_delta(rag_agent, before)
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

In [ ]:
query = "Which hotels in Cairo have both a spa and a swimming pool, and what are their guest ratings?"

print("[GRAPH-RAG]")
before = usage_snapshot(graph_agent)
r = graph_agent(query)

if r.metrics:
    usage = usage_delta(graph_agent, before)
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 RAG finds partial matches, cannot filter by multiple criteria | Graph-RAG traverses Hotel→Amenity→Amenity with AND logic")

In [ ]:
query = "Tell me about hotels in Antarctica"

print("\n[GRAPH-RAG]")
before = usage_snapshot(graph_agent)
r_graph = graph_agent(query)

if r_graph.metrics:
    usage = usage_delta(graph_agent, before)
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 RAG's honesty here rides on the model, not a guarantee | Graph-RAG returns 'No hotels found' by construction")